# Solve And Visualize A Processed MILP Instance

This notebook loads a processed instance, solves the MILP with Gurobi, validates the selected schedule, and visualizes the solution.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import load_processed_inputs
from src.data.validation import validate_clusters, validate_hourly_inputs, validate_jobs
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.plotting import plot_cluster_loads, plot_hourly_profiles, plot_schedule_gantt
from src.evaluation.results import extract_cluster_hourly_results, extract_hourly_results, extract_schedule
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model

## Configuration

Change `DATA_DIR` to solve another instance folder with the same file names: `jobs.csv`, `hourly_inputs.csv`, `clusters.csv`, and `config.json`.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TIME_LIMIT_SECONDS = 60
MIP_GAP = None

print(DATA_DIR)

## Load And Validate Inputs

In [ ]:
jobs_df, hourly_df, clusters_df, config = load_processed_inputs(DATA_DIR)

validate_jobs(jobs_df)
validate_hourly_inputs(hourly_df)
validate_clusters(clusters_df)

print(f"Jobs: {len(jobs_df)}")
display(jobs_df.groupby("category").size().rename("count").reset_index())
display(clusters_df)
display(hourly_df.head())
config

## Build And Solve

In [ ]:
model, variables = build_milp_model(jobs_df, hourly_df, clusters_df, config)
model.Params.OutputFlag = 1

solve_model(model, time_limit=TIME_LIMIT_SECONDS, mip_gap=MIP_GAP)

print(f"Gurobi status: {model.Status}")
print(f"Solutions found: {model.SolCount}")
print(f"Objective value: {model.ObjVal:,.6f}")

Status note: if `TIME_LIMIT_SECONDS` is reached after Gurobi finds a feasible solution, the notebook still extracts and visualizes the incumbent schedule. Remove or increase the time limit to prove optimality.

## Extract Results

In [ ]:
schedule_df = extract_schedule(jobs_df, variables)
hourly_results = extract_hourly_results(hourly_df, variables)
cluster_hourly_results = extract_cluster_hourly_results(variables)
metrics = compute_summary_metrics(hourly_results, config)

display(schedule_df)
display(hourly_results)
display(cluster_hourly_results.head())

## Validation Checks

In [ ]:
cluster_categories = {
    row.cluster_id: {item.strip() for item in str(row.compatible_categories).split(",")}
    for row in clusters_df.itertuples(index=False)
}

schedule_with_checks = schedule_df.copy()
schedule_with_checks["compatible"] = schedule_with_checks.apply(
    lambda row: row["category"] in cluster_categories[row["assigned_cluster"]], axis=1
)

capacity_violations = cluster_hourly_results[
    cluster_hourly_results["cluster_load"] > cluster_hourly_results["capacity"] + 1e-8
]

print(f"All jobs assigned once: {len(schedule_df) == len(jobs_df) and schedule_df['job_id'].nunique() == len(jobs_df)}")
print(f"All assignments compatible: {schedule_with_checks['compatible'].all()}")
print(f"Cluster capacity violations: {len(capacity_violations)}")
display(schedule_with_checks.loc[~schedule_with_checks["compatible"]])
display(capacity_violations)

## Metrics

In [ ]:
for key, value in metrics.items():
    print(f"{key}: {value:,.6f}")

display(schedule_df.groupby(["category", "assigned_cluster"]).size().rename("jobs").reset_index())

## Visualize Solution

In [ ]:
fig = plot_hourly_profiles(hourly_results)

In [ ]:
fig = plot_cluster_loads(cluster_hourly_results)

In [ ]:
fig = plot_schedule_gantt(schedule_df)

## Optional Export

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "processed_instance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

schedule_df.to_csv(OUTPUT_DIR / "schedule.csv", index=False)
hourly_results.to_csv(OUTPUT_DIR / "hourly_results.csv", index=False)
cluster_hourly_results.to_csv(OUTPUT_DIR / "cluster_hourly_results.csv", index=False)

print(f"Saved outputs to {OUTPUT_DIR}")